In [43]:
!pip install clustering-benchmarks -q

In [44]:
import clustbench
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"

In [45]:
battery_datasets_dict = {
                         'fcps': ['atom', 'chainlink', 'engytime', 'hepta', 'lsun', 'target', 'tetra', 'twodiamonds', 'wingnut'],
                         'uci': ['ecoli', 'glass', 'ionosphere', 'sonar', 'statlog', 'wdbc', 'wine', 'yeast'],
                         #'mnist': ['digits', 'fashion'], # Genie digits ~ 32 min
                         #'sipu': ['worms_64'], # Genie ~ 9 min
                         }

## Create autoencoder using pytorch

In [46]:
import torch
import torch.nn as nn
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

def autoencoder_feat_eng(X, n_embeddings=8):

  X = StandardScaler().fit_transform(X)
  X = torch.tensor(X, dtype=torch.float)

  # --- Autoencoder definition ---
  class Autoencoder(nn.Module):
      def __init__(self, input_dim, latent_dim=n_embeddings):
          super().__init__()
          self.encoder = nn.Sequential(
              nn.Linear(input_dim, 64),
              nn.ReLU(),
              nn.Linear(64, latent_dim)
          )
          self.decoder = nn.Sequential(
              nn.Linear(latent_dim, 64),
              nn.ReLU(),
              nn.Linear(64, input_dim)
          )

      def forward(self, x):
          z = self.encoder(x)
          x_hat = self.decoder(z)
          return x_hat, z

  # --- Train AE ---
  input_dim = X.shape[1] # Dynamically set input_dim
  model = Autoencoder(input_dim)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
  criterion = nn.MSELoss()

  for epoch in range(100):
      x_hat, z = model(X)
      loss = criterion(x_hat, X)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  # --- Cluster latent space ---
  with torch.no_grad():
      latent = model.encoder(X).numpy()

  return latent

### Difference between original dataset and embeddings from the autoencoder

In [47]:
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"
battery = "uci"
dataset = "statlog"
b = clustbench.load_dataset(battery, dataset, url=data_url)

In [48]:
import pandas as pd
pd.DataFrame(b.data)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,1.765320,1.035127,0.001835,-0.000090,-0.020114,-0.097887,-0.024913,-0.146013,0.428179,0.372142,0.588548,0.323848,-0.168115,0.481107,-0.312993,0.570539,-0.002047,-0.012852
1,-0.225939,0.124838,-0.000271,-0.000090,-0.030648,-0.103515,-0.039662,-0.149412,-0.685803,-0.622436,-0.789530,-0.645444,0.190104,-0.311184,0.121082,-0.807540,0.010869,-0.014419
2,1.461891,-1.562993,-0.000273,-0.000089,-0.018006,-0.093629,-0.024911,-0.136886,1.630662,1.499469,1.812803,1.579711,-0.393579,0.546428,-0.152849,1.794795,-0.004315,-0.017771
3,-1.762053,0.940305,-0.000271,-0.000089,-0.003257,-0.074485,0.124696,-0.028337,0.124045,0.127711,0.165011,0.079418,0.010995,0.122889,-0.133885,0.147001,-0.003034,-0.012061
4,-1.212087,1.395451,-0.000273,-0.000088,-0.008524,-0.079537,0.003536,-0.119821,0.237832,0.216212,0.329366,0.167920,-0.064862,0.274605,-0.209743,0.311358,-0.002351,-0.012505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2305,-1.799983,-0.406165,-0.000272,-0.000090,-0.012739,-0.106024,-0.020697,-0.141165,-0.318456,-0.236827,-0.363886,-0.354654,0.244889,-0.136290,-0.108598,-0.381896,-0.000870,-0.003644
2306,0.342993,-1.885387,-0.000272,-0.000090,-0.011685,-0.091065,-0.029127,-0.134702,1.717756,1.609040,1.848625,1.695604,-0.326150,0.392605,-0.066455,1.830617,-0.004883,-0.018707
2307,-0.851763,-0.975097,-0.000273,-0.000089,-0.012740,-0.089240,-0.018590,-0.134196,0.416238,0.351071,0.573796,0.323849,-0.195507,0.472678,-0.277171,0.555789,-0.002129,-0.013793
2308,-0.510405,0.181730,-0.000272,-0.000089,-0.025383,-0.105008,-0.038609,-0.150121,-0.684399,-0.622437,-0.785315,-0.645443,0.185889,-0.302756,0.116868,-0.803326,0.010868,-0.014420


In [49]:
pd.DataFrame(autoencoder_feat_eng(b.data))

,0,1,2,3,4,5,6,7
0,-0.887853,-0.512796,-0.478173,1.744358,-0.416530,0.710870,-1.242558,0.335504
1,-0.245098,1.026281,0.256429,-1.786470,1.266152,-1.137066,-0.380373,0.384247
2,0.660127,-2.201890,-1.053666,3.686686,0.737069,1.174350,-0.630006,1.229829
3,0.654528,0.689983,-0.493351,0.519271,-0.287148,0.406165,-0.009526,0.877919
4,0.456648,0.031729,-0.328218,0.835000,-0.058149,0.064469,-0.315677,0.998885
...,...,...,...,...,...,...,...,...
2305,0.066636,0.629598,0.357993,-0.261836,0.676942,-0.698187,-0.013893,0.467560
2306,0.622111,-2.071736,-0.915951,3.849685,0.838584,1.027400,-0.441607,1.088569
2307,0.702736,-0.655596,-0.473942,1.524559,0.555005,0.292966,-0.350137,0.988935
2308,-0.201299,1.047749,0.278978,-1.803555,1.253175,-1.198638,-0.347634,0.440879


## Function get_scores

Get the NCA score of a specific dataset using genie mst algorithm

In [50]:
import genieclust
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores(battery, dataset, apply_scale=False):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = b.data
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

## Function get_scores_with_autoencoder

Get the NCA score of a specific dataset applying the autoencoder transformation to the data and then using genie mst algorithm

In [51]:
import genieclust
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores_with_autoencoder(battery, dataset, apply_scale=False, n_embeddings=2):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = autoencoder_feat_eng(b.data, n_embeddings=n_embeddings)

  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

## Execute get_scores on all datasets as baseline

In [52]:
import tqdm
import pandas as pd
columns = ['Battery', 'Dataset', 'Genie NCA Score']
df = pd.DataFrame(columns=columns)
scores_lists = {}
for col in columns:
  scores_lists[col] = []

for battery in tqdm.tqdm(battery_datasets_dict.keys(), desc="Processing Datasets"):
  for dataset in battery_datasets_dict[battery]:
    scores_lists['Battery'].append(battery)
    scores_lists['Dataset'].append(dataset)
    scores_lists['Genie NCA Score'].append(get_scores(battery, dataset))

df = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 2/2 [00:05<00:00,  2.67s/it]


In [53]:
df

,Battery,Dataset,Genie NCA Score
0,fcps,atom,1.000000
1,fcps,chainlink,1.000000
2,fcps,engytime,0.918870
3,fcps,hepta,1.000000
4,fcps,lsun,1.000000
5,fcps,target,1.000000
6,fcps,tetra,1.000000
7,fcps,twodiamonds,0.992500
8,fcps,wingnut,1.000000
9,uci,ecoli,0.435664


## Execute get_scores_with_autoencoder on all datasets trying different numbers of embeddings (last hidden layer of the neural network)



In [54]:
n_embeddings_list = [2,4,16,32,64,128]

scores_lists = {}
for n_embs in tqdm.tqdm(n_embeddings_list, desc="Processing Datasets"):
  for battery in battery_datasets_dict.keys():
    for dataset in battery_datasets_dict[battery]:
      column_name = 'Genie+Autoencoder '+ str(n_embs) +' NCA Score'
      if column_name not in scores_lists:
        scores_lists[column_name] = []
      scores_lists[column_name].append(get_scores_with_autoencoder(battery, dataset, n_embeddings=n_embs))

df_ssnn = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 6/6 [00:51<00:00,  8.50s/it]


# Final Results

In [56]:
df = pd.concat([df, df_ssnn], axis=1)
numerical_cols = df.columns[2:]
df.style.highlight_max(axis=1, subset=numerical_cols)

,Battery,Dataset,Genie NCA Score,Genie+Autoencoder 2 NCA Score,Genie+Autoencoder 4 NCA Score,Genie+Autoencoder 16 NCA Score,Genie+Autoencoder 32 NCA Score,Genie+Autoencoder 64 NCA Score,Genie+Autoencoder 128 NCA Score
0,fcps,atom,1.000000,0.505000,1.000000,1.000000,1.000000,1.000000,1.000000
1,fcps,chainlink,1.000000,0.788000,1.000000,1.000000,1.000000,1.000000,1.000000
2,fcps,engytime,0.918870,0.946258,0.940378,0.917892,0.921314,0.952110,0.918875
3,fcps,hepta,1.000000,0.761111,1.000000,1.000000,1.000000,1.000000,1.000000
4,fcps,lsun,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
5,fcps,target,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
6,fcps,tetra,1.000000,0.510000,1.000000,0.996667,0.996667,0.996667,0.570000
7,fcps,twodiamonds,0.992500,0.990000,0.990000,0.990000,0.990000,0.990000,0.532500
8,fcps,wingnut,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
9,uci,ecoli,0.435664,0.361610,0.470958,0.390146,0.371743,0.510490,0.298980
